## Monitorage de l'utilisation des ressources

Il est essentiel de veiller à ce que vos tâches utilisent efficacement les ressources qui leur sont allouées. Cela est particulièrement vrai lorsque vous utilisez un nouveau programme ou que vous avez apporté une modification importante au travail effectué par votre tâche. Ce chapitre décrit différentes méthodes d'évaluation de l'efficacité des tâches, qu'elles soient en cours d'exécution ou terminées.

### Monitorage des tâches via la ligne de commande

#### Taches actuels
Par défaut, la file d'attente affichera toutes les tâches gérées par le planificateur. Elle sera beaucoup plus rapide si vous ne consultez que vos propres tâches.

`squeue -u $USER`

Vous pouvez afficher uniquement les tâches en cours ou uniquement les tâches en attente :

`squeue -u <username> -t RUNNING` <br>
`squeue -u <username> -t PENDING`

Vous pouvez afficher des informations détaillées pour un emploi spécifique avec scontrol :

`scontrol show job -dd JOB_ID`

N’exécutez pas la commande squeue à haute fréquence (par exemple, toutes les quelques secondes) depuis un script ou un programme. Répondre à squeue surcharge Slurm et peut nuire à ses performances ou à son bon fonctionnement.

#### Taches terminés
Vous pouvez obtenir un bref résumé de l'efficacité du processeur et de la mémoire d'une tâche avec `seff` :

`seff JOB_ID`

Des informations plus détaillées sur un travail terminé peuvent être obtenues avec `sacct` :

`sacct -j JOB_ID`

#### Se connecter à une tâche en cours

Il est possible de se connecter au nœud exécutant une tâche et d'y lancer de nouveaux processus. Cela peut s'avérer utile pour le dépannage ou le suivi de la progression d'une tâche.

Supposons que vous souhaitiez exécuter l'utilitaire nvidia-smi pour surveiller l'utilisation du GPU sur un nœud où une tâche est en cours d'exécution. La commande suivante lance une surveillance sur le nœud affecté à la tâche, ce qui exécute nvidia-smi toutes les 30 secondes et affiche le résultat dans votre terminal :

`srun --jobid 31 --overlap --pty watch -n 30 nvidia-smi`

Il est possible de lancer plusieurs commandes de surveillance avec tmux. La commande suivante lance htop et nvidia-smi dans des fenêtres séparées pour surveiller l'activité sur un nœud affecté à la tâche spécifiée :

`srun --jobid 31 --overlap --pty tmux new-session -d 'htop -u $USER' \; split-window -h 'watch nvidia-smi' \; attach`

**Attention :** les processus lancés avec `srun` partagent les ressources avec la tâche spécifiée. Il convient donc de veiller à ne pas lancer de processus susceptibles d'utiliser une part importante des ressources allouées à la tâche. Par exemple, une utilisation excessive de la mémoire peut entraîner l'arrêt brutal de la tâche ; une utilisation excessive du processeur la ralentira. Supposons que vous souhaitiez exécuter l'utilitaire `nvidia-smi` pour surveiller l'utilisation du GPU sur un nœud où une tâche est en cours d'exécution. La commande suivante lance `watch` sur le nœud affecté à la tâche, ce qui exécute à son tour `nvidia-smi` toutes les 30 secondes et affiche le résultat dans votre terminal :

### Monitorage des processus CPU
#### Top
La commande `top` est un outil de surveillance système en temps réel préinstallé sur la plupart des distributions Linux. Elle offre une vue dynamique des ressources système, notamment une liste en direct des processus en cours d'exécution.

**Start Monitoring**: Ouvrez votre terminal et tapez top, puis appuyez sur Entrée. <br>
**Exit**: Appuyez sur q pour quitter l'interface et revenir à la ligne de commande. 

![cpu-vs-cpu-bandwitdh](images/top.png)

<u>*Les métriques du CPU*</u>

%Cpu(s) Line: <br>
- us (User): Temps consacré aux processus au niveau utilisateur (par exemple, les applications)
- sy (System): Temps passé sur les processus du noyau
- id (Idle): Le temps pendant lequel le processeur est inactif. Un temps d'inactivité élevé signifie une faible charge
- wa (I/O Wait): Temps passé par le processeur à attendre des opérations sur disque ou réseau

<u>*Options de démarrage spécialisées*</u>

Vous pouvez lancer la commande `top` avec des paramètres spécifiques pour l'automatisation ou la surveillance ciblée :
* Surveiller un utilisateur spécifique : `top -u [nom_utilisateur]`.
* Surveiller des PID spécifiques : `top -p [PID1],[PID2]`.
* Mode batch (instantanés) : Utilisez `top -b -n 1 > snapshot.txt` pour capturer un instantané de l'état du système dans un fichier, sans interface interactive.

#### Htop

Htop est un système de monitorage interactif en temps réel et un gestionnaire de processus pour les systèmes de type Unix. Il constitue une alternative améliorée et conviviale à la commande top traditionnelle. Il affiche l'utilisation du processeur et de la mémoire, fournit des indicateurs de performance codés par couleur et permet de gérer les processus (kill, renice) à l'aide des touches fléchées et des touches de fonction.


![cpu-vs-cpu-bandwitdh](images/htop.png)

<u>*Key Interactive Features*</u>

- Navigation : Utilisez les flèches directionnelles pour faire défiler les processus verticalement ou horizontalement.
- Sorting: Appuyez sur F6 pour trier par processeur, mémoire ou PID.
- Search: Appuyez sur F3 pour rechercher un processus spécifique.
- Manage Processes: Appuyez sur F9 pour arrêter un processus sélectionné ou lui envoyer des signaux.
- Tree View: Appuyez sur F5 pour afficher les processus sous forme d’arborescence hiérarchique.
- Colors: Des barres de couleur permettent de visualiser rapidement l’état des processus.

### Monitorage des processus GPU
#### Nvidia-smi

Nous allons commencer par les outils de surveillance. La première chose à vérifier est si votre code utilise bien le GPU. La méthode la plus simple pour répondre à cette question est d'utiliser la commande `nvidia-smi`.

![cpu-vs-cpu-bandwitdh](images/nvidia-smi.png)

Comme vous pouvez le voir ci-dessus, <code>nvidia-smi</code> vous indiquera si des processus utilisent actuellement le GPU et quelle quantité de mémoire GPU ils ont allouée.

Une erreur fréquente d'interprétation du champ <code>utilisation %</code> consiste à le considérer comme la quantité de capacité de calcul actuellement utilisée. Or, ce n'est **pas** ce que représente ce nombre. En réalité, <code>utilisation %</code> correspond à la proportion de temps pendant laquelle *au moins un noyau* était exécuté sur le GPU. Un faible pourcentage signifie que le GPU est soit inactif, soit occupé à d'autres tâches que l'exécution de noyaux, comme la gestion de la mémoire ou la planification des tâches.


Idéalement, ce pourcentage reste élevé pendant toute la durée de votre tâche, mais **cela ne suffit pas** à conclure que votre code n'utilise pas inutilement la capacité du GPU. Vous pourriez en effet avoir un seul noyau fonctionnant en permanence avec seulement quelques cœurs, gaspillant ainsi massivement la capacité du GPU.

#### Nvtop

Une autre façon de vérifier si votre code utilise effectivement le GPU est d'utiliser la commande `nvtop`. Cet outil affiche les mêmes statistiques d'utilisation que nvidia-smi sous forme de graphique linéaire évoluant dans le temps. Vous obtiendrez également une ventilation du pourcentage d'utilisation par processus si plusieurs processus partagent le même GPU.

![cpu-vs-cpu-bandwitdh](images/nvtop.png)

Le graphique linéaire permet d'identifier les pics et les variations du taux d'utilisation de la mémoire, ainsi que toute modification de l'allocation mémoire pendant l'exécution de votre code. Cela peut vous fournir des indications précieuses pour optimiser votre code et minimiser, par exemple, les temps d'inactivité.

Un autre indicateur intéressant fourni par cet outil est le **CPU % utilisation**. En général, l'utilisation du processeur doit être faible pour un programme utilisant le GPU. Un pourcentage d'utilisation élevé peut indiquer que votre programme n'exploite pas le GPU de manière optimale, car une grande partie du travail est effectuée par le processeur.

### Monitorage des ressources avec des portails des grappes de calul
Nous allons maintenant nous intéresser à un outil qui se situe à la frontière entre la surveillance et le profilage : les portails de regroupement de l’Alliance de recherche numérique du Canada.

<u>*Monitoring CPU metrics*</u>
![cpu-vs-cpu-bandwitdh](images/portal1.png)

![cpu-vs-cpu-bandwitdh](images/portal2.png)

<u>*Monitoring whole node resources*</u>
![cpu-vs-cpu-bandwitdh](images/portal3.png)

<u>*Monitoring GPU metrics*</u>

![cpu-vs-cpu-bandwitdh](images/portal.png)


Ci-dessus, vous pouvez voir un autre ensemble de statistiques relatives au même exemple de code :

- **CPU:** Il s'agit de la moyenne du ratio de cycles consommés sur chaque cœur de processeur par tous les processus de cette tâche. Ce graphique devrait être entièrement rempli la plupart du temps ; si ce n'est pas le cas, vous pouvez réduire le nombre de cœurs demandés au planificateur. Les cycles inutilisés n'améliorent pas les performances de votre tâche et diminueront sa priorité lorsque des cœurs sont gaspillés.

- **Memory:** La quantité maximale de mémoire utilisée devrait être proche de la mémoire allouée. Si la mémoire n'est pas utilisée par la tâche, réduisez la quantité allouée ; vos tâches pourront démarrer plus rapidement. La mémoire inutilisée n'améliore pas les performances de vos tâches.

- **Filesystem:** l s'agit de l'utilisation du système de fichiers par la tâche, et non par le nœud entier.

- **SM Activity:** Il s'agit du pourcentage moyen de temps, sur l'ensemble des SM, où au moins un ensemble d'opérations simultanées était actif, quel que soit le nombre de noyaux concernés. **Notez que** « actif » ne signifie pas « calcul en cours ». D'autres activités, comme l'attente de données et l'accès à la mémoire, sont également prises en compte. Idéalement, ce pourcentage devrait rester supérieur à 80 % pendant toute la durée de votre tâche, mais cela **ne signifie pas nécessairement** que vous utilisez 80 % de la capacité de calcul du GPU. Cela signifie soit que vous avez utilisé 100 % de la capacité pendant 80 % du temps et que le GPU est resté inactif pendant 20 % du temps, soit que vous avez utilisé 80 % de la capacité pendant 100 % du temps, avec 20 % de capacité disponible..

<br>

- **SM Occupancy:**  Il s'agit de la moyenne, sur la durée, du ratio entre les opérations simultanées actives et le nombre maximal d'opérations simultanées prises en charge par le GPU. Un pourcentage élevé **ne signifie pas nécessairement** que vous utilisez efficacement le GPU. Toutefois, pour nos besoins, nous considérerons qu'un pourcentage élevé est un bon indicateur d'une utilisation optimale..

<br>

Les autres indicateurs affichés ci-dessus — Tensor, FP64, FP32 et FP16 — indiquent le type de cœur GPU utilisé et le pourcentage de temps d'activité de ces parties spécifiques de la carte. L'indicateur Tensor mérite une attention particulière. Les cœurs Tensor sont des cœurs spécialisés conçus pour accélérer les opérations sur les tenseurs n-dimensionnels, telles que les multiplications et les convolutions. 

L'utilisation élevée des cœurs Tensor devrait se traduire par des gains de vitesse significatifs par rapport aux autres types de cœurs GPU. **Attention toutefois :** les deux premières conditions impliquent une perte de précision dans les calculs. Il est donc important de bien évaluer l'importance d'une haute précision pour votre application.

### Les portals disponible sur Alliance

L'alliance gère actuellement des portails pour les clusters suivants :

- [Rorqual](https://metrix.rorqual.calculquebec.ca)
- [Narval](https://portail.narval.calculquebec.ca)
- [Nibi](https://portal.nibi.sharcnet.ca)
- [Tamia](https://portail.tamia.ecpia.ca)
- [Trillium](https://my.scinet.utoronto.ca)

# Example:  CIFAR10 data set

Cet exemple illustre le monitorage des ressources CPU/GPU à l'aide de portails et d'outils simples tels que `top`, `htop`, `nvtop` et `nvidia-smi`.

Vous utiliserez PyTorch pour entraîner un réseau de neurones convolutif sur le ensemble de données CIFAR10, une collection de 60 000 images représentant 10 animaux et modes de transport différents. Ce réseau est composé d'une séquence de deux convolutions, suivies de 400 multiplications matricielles simultanées, puis de 120 multiplications matricielles simultanées et enfin de 10 multiplications matricielles simultanées. Les convolutions et les multiplications matricielles sont des opérations parallélisables.


In [ ]:
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

device = torch.device("cuda")

class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()

        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

#net = Net().cuda() # Load model on the GPU
net = Net().to(device)

#criterion = nn.CrossEntropyLoss().cuda() # Load the loss function on the GPU
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(net.parameters(), lr=0.001)

transform_train = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

Nous utilisons ici une variante de la méthode de descente de gradient appelée descente de gradient par mini-lots. Au lieu d'utiliser la matrice d'entrée entière à chaque itération de l'algorithme, on utilise un petit sous-ensemble de lignes. Le nombre de lignes composant ce sous-ensemble, appelé taille du lot, est un facteur important qui influence la convergence de cette approche, ainsi que sa rapidité.

Ensuite, vous utiliserez la descente de gradient par mini-lots pour entraîner notre réseau de neurones convolutif aux données CIFAR10. Le code ci-dessous vous permettra de charger un lot d'images à la fois depuis le disque, puis de chronométrer l'exécution de trois itérations de cet algorithme d'entraînement. Testez différentes combinaisons des trois paramètres ci-dessous et observez leur impact sur le temps d'exécution et l'utilisation du GPU :

1. **BATCH_SIZE**: Il s'agit de la taille du lot de descente de gradient par mini-lots, c'est-à-dire le nombre d'images qui doivent être chargées à partir du disque et utilisées simultanément par l'algorithme d'entraînement.
3. **NUM_WORKERS**: Ce paramètre détermine le nombre de cœurs CPU utilisés pour charger les lots d'images *en parallèle*. L'objectif est d'utiliser plusieurs processus, chacun avec un cœur CPU, afin d'optimiser le chargement des images depuis le disque *sans* bloquer le processus principal.
4. **PRE_FETCH**: Ce paramètre détermine le nombre de lots d'images à charger depuis le disque *avant* qu'ils ne soient nécessaires à l'algorithme d'entraînement. Ces lots seront stockés dans la mémoire système et chargés sur le GPU lorsqu'ils seront utilisés.

In [ ]:
f"{os.getenv('SLURM_TMPDIR')}/data"

In [ ]:
BATCH_SIZE = 4096
NUM_WORKERS = 2
PRE_FETCH = 2

#datadir = os.environ["CIFAR10_PATH"] #f"{os.getenv('SLURM_TMPDIR')}/data"
datadir = f"{os.getenv('SLURM_TMPDIR')}/data"
dataset_train = CIFAR10(root=datadir, train=True, download=False, transform=transform_train)

train_loader = DataLoader(dataset_train, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, prefetch_factor=PRE_FETCH)

In [ ]:
def train(model, n_iterations, data):
    for iteration in range(n_iterations):
        for batch_idx, (inputs, targets) in data:

#            inputs = inputs.cuda()
#            targets = targets.cuda()
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

In [ ]:
%time train(net,n_iterations=3, data=enumerate(train_loader))

Dans l'exemple ci-dessus, nous avons utilisé l'objet DataLoader pour lire itérativement des lots d'images depuis le disque, ce qui implique deux séries d'opérations de lecture/écriture : disque → mémoire système et mémoire système → mémoire GPU. L'étape disque → mémoire système est susceptible de créer des goulots d'étranglement, car la lecture depuis le disque est généralement beaucoup plus lente que la lecture depuis la mémoire. Quelle serait la différence de performance si, au lieu de cela, nous chargions l'ensemble des données dans la mémoire système avant de commencer nos calculs ?

In [ ]:
cifar10 = [(batch_idx,(inputs,targets)) for batch_idx, (inputs, targets) in enumerate(train_loader)]

%timeit train(net,n_iterations=3, data=cifar10)

Le réseau de neurones convolutif que vous avez utilisé précédemment n'était pas très volumineux. Si vous observez la quantité de mémoire GPU utilisée lors du chargement de ce modèle, vous constaterez qu'elle se situe entre 1 et 2 Go. Il reste donc largement assez d'espace pour charger le jeu de données CIFAR10. Aurions-nous pu accélérer encore davantage l'algorithme d'entraînement en chargeant simplement toutes les images dans la mémoire GPU avant d'appeler la fonction `train()` ?

In [ ]:
cifar10 = [(batch_idx,(inputs.cuda(),targets.cuda())) for batch_idx, (inputs, targets) in enumerate(train_loader)]

%timeit train(net,n_iterations=3, data=cifar10)

### Multiplication des matrices

Nous exécutons ici l'exemple de multiplication matricielle, car il s'agit d'une tâche très intensif en ressources GPU, et nous surveillons l'utilisation du GPU sur le portail.

In [ ]:
import os
import timeit

N_CPUS=4

os.environ["OMP_NUM_THREADS"] = str(N_CPUS)

import numpy as np
import cupy as cp

In [ ]:
import cupy
print(cupy.cuda.runtime.runtimeGetVersion())

In [ ]:
pip install --no-index cupy==13.6.0

In [ ]:
def generate_data(n_rows,n_cols,device):
    lib = cp if device == "gpu" else np

    rmg = lib.random.rand #random matrix generator
    a = rmg(n_rows,n_cols)
    b = rmg(n_rows,n_cols)
    return a,b

def matmul(a,b):
    lib = cp if type(a) == cp.ndarray else np

    results = lib.matmul(a,b)
    return

In [ ]:
a,b = generate_data(5000,5000,"gpu")

In [ ]:
%timeit matmul(a,b)